#  Stock Bubble Detector
### Identifying Overheated Stocks Before They Crash — Powered by Yahoo Finance

---

### What Is a Financial Bubble?

A financial bubble occurs when an asset's price rises **far above its true value**, driven by speculation rather than fundamentals. When confidence breaks, the price collapses — often wiping out years of gains in weeks.

| Bubble | Year | Peak-to-Trough Crash |
|--------|------|---------------------|
| Dot-Com | 2000 | NASDAQ fell **−78%** |
| Housing Crisis | 2008 | S&P 500 fell **−57%** |
| Crypto Mania | 2021 | Bitcoin fell **−77%** |
| Meme Stocks | 2021 | GameStop fell **−90%** |

In every case, four quantifiable warning signs appeared **before** the crash. This notebook detects all four.

### How the Score Works

| Indicator | What It Catches | Max Points |
|-----------|----------------|------------|
| RSI Analysis | Overbought momentum | 30 pts |
| Price vs 200-Day MA | How far above fair value | 30 pts |
| Volatility | Speculation and panic level | 20 pts |
| Price Acceleration | Parabolic (speeding-up) moves | 20 pts |

**Score guide:** 0–29  Healthy · 30–49  Moderate · 50–69  High · 70–100  Critical

---

##  Step 0 — Install Libraries

Run this cell once. After that you can skip it.

In [ ]:
# Install all required libraries
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'yfinance', 'pandas', 'numpy', 'matplotlib', '--quiet'
])
print('All libraries ready!')

##  Step 1 — Import the Module

We import every function from `stock_bubble_detector.py`.
The file is split into **7 sections** — each section is demonstrated below.

| Section | What It Contains |
|---------|------------------|
| 1 | `load_stock_data()` — downloads OHLCV data from Yahoo Finance |
| 2 | Metric functions — price, returns, volatility, RSI, MA, drawdown |
| 3 | Bubble detection — four checks that produce the 0–100 bubble score |
| 4 | `generate_strategies()` — avoidance plans scaled to risk level |
| 5 | Print functions — formatted console reports |
| 6 | `plot_stock_detail()` — two-panel price + RSI chart |
| 7 | `main()` — interactive CLI loop |

In [ ]:
# Standard library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Import every function from our module ────────────────────
# Make sure stock_bubble_detector.py is in the same folder
from stock_bubble_detector import (
    # Section 1 — data loading
    load_stock_data,
    # Section 2 — metrics
    get_current_price, get_total_return, get_daily_returns,
    get_volatility, get_moving_average, get_rsi,
    get_price_to_200ma_ratio, get_max_drawdown,
    # Section 3 — bubble detection
    check_rsi, check_moving_average, check_volatility,
    check_acceleration, assign_risk_level, run_bubble_analysis,
    RSI_WARNING, RSI_DANGER,
    MA_RATIO_WARNING, MA_RATIO_DANGER,
    VOLATILITY_HIGH, VOLATILITY_EXTREME,
    # Section 4 — strategies
    generate_strategies,
    # Section 5 — printing
    print_stock_summary, print_bubble_report, print_strategies,
    # Section 6 — charting
    plot_stock_detail,
)

print('Module loaded successfully.')
print(f'Analysis date: {datetime.today().strftime("%Y-%m-%d")}')

---
## Section 1 — Load Stock Data

**`load_stock_data(ticker, start_date, end_date)`** connects to Yahoo Finance
and downloads historical daily price data — no API key required.

The function returns a pandas DataFrame with five columns:
`Open`, `High`, `Low`, `Close`, `Volume`.

> **Change "TICKER" to any stock symbol you want to analyze.**

In [ ]:
# ── Choose your stock ────────────────────────────────────────
TICKER     = 'AAPL'   # Try: 'TSLA', 'NVDA', 'MSFT', 'META', 'AMD'
YEARS_BACK = 2        # How many years of data to pull (1–5)

# Calculate date range
END_DATE   = datetime.today().strftime('%Y-%m-%d')
START_DATE = (datetime.today() - timedelta(days=365 * YEARS_BACK)).strftime('%Y-%m-%d')

print(f'Ticker     : {TICKER}')
print(f'Date range : {START_DATE}  to  {END_DATE}')
print(f'Years back : {YEARS_BACK}')

In [ ]:
# Download the data
data = load_stock_data(TICKER, START_DATE, END_DATE)

# Preview — first 5 rows of the DataFrame
print(f'\nFirst 5 rows of {TICKER} price data:')
data.head()

In [ ]:
# Basic statistics of the closing price over the analysis window
print(f'{TICKER} Closing Price — Descriptive Statistics')
print('=' * 45)
print(data['Close'].describe().round(2).to_string())

The table above shows the **price range** across the analysis window.
A large gap between `min` and `max` indicates high volatility —
prices swung a lot, which is one of our bubble indicators.

The `mean` represents the average price over the period.
If the current price (the `max` or near it) is far above the mean,
that suggests the stock has run up significantly.

---
## Section 2 — Metric Calculations

Before scoring for bubbles, we calculate six core metrics.
These are the same numbers shown on every professional trading terminal.

### 2.1 — Key Metrics at a Glance

In [ ]:
# Print the full metric summary using Section 5's print function
print_stock_summary(TICKER, data)

**What each metric means:**

- **Current Price** — the most recent closing price
- **Total Return** — percentage gain or loss since `START_DATE`
- **Annualized Volatility** — how much the price swings per year
  *(S&P 500 averages ~15%; anything above 40% is considered high)*
- **Max Drawdown** — the worst peak-to-trough fall in the window
  *(tells you the worst-case loss if you bought at the top)*
- **RSI (14-day)** — momentum indicator; above 70 = overbought
- **Price / 200-Day MA** — above 1.30× means 30% above fair value; 1.50× = danger zone

### 2.2 — Explore Metrics Individually

In [ ]:
# Each metric function is independently callable
price   = get_current_price(data)
ret     = get_total_return(data)
vol     = get_volatility(data)
dd      = get_max_drawdown(data)
rsi_now = float(get_rsi(data).iloc[-1])
ratio   = get_price_to_200ma_ratio(data)

print(f'Metric breakdown for {TICKER}:')
print(f'  get_current_price()         → ${price}')
print(f'  get_total_return()          → {ret:+.2f}%')
print(f'  get_volatility()            → {vol:.2f}% annualized')
print(f'  get_max_drawdown()          → {dd:.2f}%')
print(f'  get_rsi() [latest value]    → {rsi_now:.1f}')
if ratio is not None:
    print(f'  get_price_to_200ma_ratio()  → {float(ratio.iloc[-1]):.3f}x')

### 2.3 — Daily Returns Distribution

The histogram below shows how daily price changes are distributed.
A wide spread means high volatility — prices swing up and down a lot.
A narrow cluster near zero means a stable, low-risk stock.

In [ ]:
daily_ret = get_daily_returns(data) * 100   # Convert to percentage

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(daily_ret, bins=55, color='#1565C0', edgecolor='black',
        alpha=0.75, label='Daily Returns')

mu, sigma = float(daily_ret.mean()), float(daily_ret.std())
ax.axvline(mu,          color='#FF9800', lw=2,   linestyle='--',
           label=f'Mean: {mu:.3f}%')
ax.axvline(mu - 2*sigma, color='#F44336', lw=1.4, linestyle=':',
           label=f'±2σ: {mu-2*sigma:.2f}% / {mu+2*sigma:.2f}%')
ax.axvline(mu + 2*sigma, color='#F44336', lw=1.4, linestyle=':')
ax.axvline(0, color='black', lw=0.8, alpha=0.5)

ax.set_xlabel('Daily Return (%)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'{TICKER} — Daily Return Distribution\n'
             f'Mean: {mu:.3f}%   Std Dev: {sigma:.3f}%',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{TICKER}_return_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Std Dev: {sigma:.3f}%  |  Best day: {float(daily_ret.max()):+.2f}%  |  '
      f'Worst day: {float(daily_ret.min()):+.2f}%')

### 2.4 — RSI Over Time

The RSI (Relative Strength Index) is a momentum indicator that oscillates 0–100.

- **Above 70** → the stock has risen too fast relative to its recent history (overbought)
- **Below 30** → the stock has fallen too fast (oversold)
- **40–60** → normal, balanced range

Extended periods above 70 are one of the most reliable early bubble signals.

In [ ]:
rsi_series = get_rsi(data)

fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(rsi_series.index, rsi_series,
        color='#7B1FA2', linewidth=1.5, label='RSI (14-day)')
ax.axhline(70, color='#F44336', linestyle='--', lw=1.3, label='Overbought (70)')
ax.axhline(30, color='#388E3C', linestyle='--', lw=1.3, label='Oversold (30)')
ax.axhline(50, color='gray',    linestyle=':',  lw=0.8)

ax.fill_between(rsi_series.index, 70, rsi_series,
                where=(rsi_series > 70), alpha=0.25, color='#F44336',
                label='Overbought zone')
ax.fill_between(rsi_series.index, 30, rsi_series,
                where=(rsi_series < 30), alpha=0.25, color='#388E3C',
                label='Oversold zone')

ax.set_ylim(0, 100)
ax.set_ylabel('RSI', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_title(f'{TICKER} — RSI (14-Day) Over Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{TICKER}_rsi.png', dpi=150, bbox_inches='tight')
plt.show()

# How many of the last 30 days were overbought?
recent_ob = int((rsi_series.tail(30) > 70).sum())
print(f'Days with RSI > 70 in last 30 trading days: {recent_ob}/30')

### 2.5 — Price vs Moving Averages

Moving averages smooth out daily noise to reveal the underlying trend.

- **50-Day MA** (orange dashed) — short-term trend
- **200-Day MA** (red dash-dot) — long-term "fair value" baseline

When the price trades **significantly above** the 200-day MA for extended periods,
it signals that the market may have gotten ahead of itself.
The red-shaded zone marks where price exceeded **130% of the 200-day MA**.

In [ ]:
prices = data['Close']
ma_50  = get_moving_average(data, 50)
ma_200 = get_moving_average(data, 200)
ratio  = get_price_to_200ma_ratio(data)

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(prices.index, prices, color='#1565C0', lw=1.3, label='Close Price')
ax.plot(ma_50.index,   ma_50,  color='#FF9800', lw=1.5,
        linestyle='--',  label='50-Day MA')
ax.plot(ma_200.index,  ma_200, color='#F44336', lw=1.8,
        linestyle='-.', label='200-Day MA')

if ratio is not None:
    bubble_zone = ratio > MA_RATIO_WARNING
    ax.fill_between(prices.index, prices, ma_200,
                    where=(bubble_zone & prices.notna() & ma_200.notna()),
                    alpha=0.12, color='red', label='Danger Zone (>30% above 200MA)')

ax.set_ylabel('Price (USD)', fontsize=12)
ax.set_title(f'{TICKER} — Price with 50-Day and 200-Day Moving Averages',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{TICKER}_ma.png', dpi=150, bbox_inches='tight')
plt.show()

if ratio is not None:
    current_ratio = float(ratio.iloc[-1])
    print(f'Current price is {(current_ratio-1)*100:.1f}% above the 200-day MA')
    if current_ratio > MA_RATIO_DANGER:
        print('Status: DANGER — exceeds 50% threshold')
    elif current_ratio > MA_RATIO_WARNING:
        print('Status: WARNING — exceeds 30% threshold')
    else:
        print('Status: OK — within normal range')

### 2.6 — Drawdown Chart

The drawdown chart shows how far the stock has fallen from its **rolling peak**
at each point in time. This makes the true downside risk tangible:
instead of an abstract volatility percentage, you see the actual loss
a peak-buyer would have experienced.

In [ ]:
prices   = data['Close']
peak     = prices.cummax()
drawdown = (prices - peak) / peak * 100

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(drawdown.index, drawdown, 0,
                alpha=0.45, color='#D32F2F', label='Drawdown')
ax.plot(drawdown.index, drawdown, color='#B71C1C', lw=1.2)
ax.axhline(0, color='black', lw=0.8, alpha=0.5)

# Mark the maximum drawdown point
max_dd_idx = drawdown.idxmin()
max_dd_val = float(drawdown.min())
ax.annotate(f'Max DD: {max_dd_val:.1f}%',
            xy=(max_dd_idx, max_dd_val),
            xytext=(20, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, fontweight='bold')

ax.set_ylabel('Drawdown from Peak (%)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_title(f'{TICKER} — Drawdown from Rolling Peak',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{TICKER}_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Maximum drawdown in the period: {max_dd_val:.2f}%')

---
## Section 3 — Bubble Detection: Running the Four Checks

Each of the four checks runs independently and contributes points
to the overall bubble score. We can inspect them one at a time
before calling `run_bubble_analysis()` which runs all four.

### 3.1 — Individual Check Results

In [ ]:
print(f'\nRunning individual bubble checks on {TICKER}...\n')

rsi_score,   rsi_warns   = check_rsi(data)
ma_score,    ma_warns    = check_moving_average(data)
vol_score,   vol_warns   = check_volatility(data)
accel_score, accel_warns = check_acceleration(data)

total = min(100, rsi_score + ma_score + vol_score + accel_score)

checks = [
    ('RSI Check',          rsi_score,   30, rsi_warns),
    ('Moving Average',     ma_score,    30, ma_warns),
    ('Volatility',         vol_score,   20, vol_warns),
    ('Price Acceleration', accel_score, 20, accel_warns),
]

print(f'  {"CHECK":<22} {"SCORE":>6}  {"MAX":>4}  FLAGS')
print(f'  {"-"*22} {"-"*6}  {"-"*4}  {"-"*30}')
for name, score, max_pts, warns in checks:
    flag = warns[0][:45] + '...' if warns else 'None'
    print(f'  {name:<22} {score:>5}  {max_pts:>4}  {flag}')
print(f'  {"─"*60}')
print(f'  {"TOTAL BUBBLE SCORE":<22} {total:>5}  {100:>4}')
print(f'  RISK LEVEL: {assign_risk_level(total)}')

### 3.2 — Full Bubble Report

`run_bubble_analysis()` combines all four checks and returns a results dictionary.
`print_bubble_report()` formats it for easy reading.

In [ ]:
# Run the full analysis
result = run_bubble_analysis(TICKER, data)

# Print the formatted report
print_bubble_report(result)

### 3.3 — Bubble Score Breakdown Chart

This chart shows exactly how much each indicator contributed to the score
and how much room is left before each reaches its maximum.

In [ ]:
indicators = list(result['sub_scores'].keys())
scores     = list(result['sub_scores'].values())
max_pts    = [30, 30, 20, 20]

def score_color(s, mx):
    pct = s / mx
    if pct >= 0.85: return '#D32F2F'
    if pct >= 0.50: return '#F57C00'
    if pct >= 0.25: return '#FBC02D'
    return '#388E3C'

fig, ax = plt.subplots(figsize=(10, 5))

# Background bars showing maximum possible points
ax.barh(indicators, max_pts, color='#EEEEEE',
        edgecolor='#BDBDBD', linewidth=0.8, label='Max Possible')

# Actual score bars
colors = [score_color(s, m) for s, m in zip(scores, max_pts)]
ax.barh(indicators, scores, color=colors,
        edgecolor='black', linewidth=0.6, alpha=0.88, label='Actual Score')

# Score labels
for i, (score, mx) in enumerate(zip(scores, max_pts)):
    ax.text(score + 0.3, i, f'{score} / {mx}',
            va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 35)
ax.set_xlabel('Points', fontsize=12)
ax.set_title(
    f'{TICKER} — Bubble Score Breakdown\n'
    f'Total: {result["bubble_score"]} / 100   |   {result["risk_level"]}',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{TICKER}_score_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Avoidance Strategies

`generate_strategies()` returns a list of recommendations scaled to the bubble score:

| Score Range | Action Level | What to Expect |
|-------------|-------------|----------------|
| 70–100 |  URGENT | Reduce position 50–75%, set stop-loss, hedge with puts |
| 50–69 |  HIGH | Trim 25–40%, set 15% trailing stop |
| 30–49 |  MEDIUM | Monitor weekly, do not add to position |
| 0–29 |  LOW | Hold or dollar-cost average |

Every strategy includes a **priority level** and a specific **actionable detail**.

In [ ]:
current_price = get_current_price(data)
strategies    = generate_strategies(TICKER, current_price, result['bubble_score'])

print_strategies(TICKER, result['bubble_score'], strategies)

In [ ]:
# We can also inspect the strategies as a list of dictionaries
print(f'Total strategies generated: {len(strategies)}\n')
for i, strat in enumerate(strategies, 1):
    print(f'  {i}. [{strat["priority"]:^6}] {strat["action"]}')

---
## Section 5 — Price & RSI Chart

`plot_stock_detail()` creates a two-panel chart:

- **Top panel** — closing price with the 50-day MA (orange dashed)
  and 200-day MA (red dash-dot). The red-shaded zone appears wherever
  the price is more than 30% above the 200-day MA.

- **Bottom panel** — RSI with overbought (70) and oversold (30) reference lines.
  Red shading = overbought zone. Green shading = oversold zone.

This is the same chart layout used by professional charting software
like TradingView and Bloomberg Terminal.

In [ ]:
plot_stock_detail(TICKER, data)

---
## Section 6 — Multi-Stock Comparison

We can run the analysis on several stocks at once and compare their
bubble scores side by side — a useful way to rank risk across a watchlist.

> **Edit `WATCHLIST` to include any tickers you want to compare.**

In [ ]:
# ── Edit this list with any tickers you want ────────────────
WATCHLIST = ['AAPL', 'TSLA', 'NVDA', 'MSFT', 'META', 'SPY']

# Download and analyze each ticker
watchlist_results = []

for ticker in WATCHLIST:
    wl_data = load_stock_data(ticker, START_DATE, END_DATE)
    if wl_data is None:
        print(f'  Skipping {ticker} — could not load data')
        continue

    wl_result = run_bubble_analysis(ticker, wl_data)
    wl_result['total_return']  = get_total_return(wl_data)
    wl_result['volatility']    = get_volatility(wl_data)
    wl_result['current_price'] = get_current_price(wl_data)
    watchlist_results.append(wl_result)

# Sort by bubble score (highest = most dangerous)
watchlist_results.sort(key=lambda r: r['bubble_score'], reverse=True)

print(f'\nAnalysis complete: {len(watchlist_results)}/{len(WATCHLIST)} stocks loaded.')

### 6.1 — Leaderboard Table

In [ ]:
# Print a ranked table
print(f'\n  {"="*68}')
print(f'  BUBBLE RISK LEADERBOARD')
print(f'  {"="*68}')
print(f'  {"RK":<4} {"TICKER":<8} {"SCORE":>6} {"RETURN":>8} '
      f'{"VOLATILITY":>11}  RISK LEVEL')
print(f'  {"-"*4} {"-"*8} {"-"*6} {"-"*8} {"-"*11}  {"-"*22}')

medals = {1: '★', 2: '▲', 3: '●'}

for rank, r in enumerate(watchlist_results, 1):
    symbol = medals.get(rank, ' ')
    print(f'  {rank:<2}{symbol}  {r["ticker"]:<8} '
          f'{r["bubble_score"]:>5}/100 '
          f'{r["total_return"]:>+7.1f}% '
          f'{r["volatility"]:>10.1f}%  '
          f'{r["risk_level"]}')

print(f'  {"="*68}')

### 6.2 — Bubble Score Comparison Chart

In [ ]:
# Color each bar by risk level
def bar_color(score):
    if score >= 70: return '#D32F2F'
    if score >= 50: return '#F57C00'
    if score >= 30: return '#FBC02D'
    return '#388E3C'

tickers_wl = [r['ticker']       for r in watchlist_results]
scores_wl  = [r['bubble_score'] for r in watchlist_results]
colors_wl  = [bar_color(s)      for s in scores_wl]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(tickers_wl, scores_wl,
              color=colors_wl, edgecolor='black', linewidth=0.7,
              alpha=0.88, width=0.6)

# Score labels on top of bars
for bar, score in zip(bars, scores_wl):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            str(score), ha='center', va='bottom',
            fontweight='bold', fontsize=12)

# Risk zone lines
for y, label, c in [(70, 'Critical (70)', '#D32F2F'),
                     (50, 'High (50)',     '#F57C00'),
                     (30, 'Moderate (30)','#FBC02D')]:
    ax.axhline(y, color=c, linestyle='--', lw=1.3, alpha=0.8, label=label)

ax.set_ylim(0, 115)
ax.set_xlabel('Stock Ticker', fontsize=12)
ax.set_ylabel('Bubble Score (0–100)', fontsize=12)
ax.set_title('Bubble Score Comparison — Watchlist',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig('watchlist_bubble_scores.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 — Risk vs Return Scatter Plot

This chart plots every stock by **volatility** (X-axis, risk) vs **total return** (Y-axis, reward).
The dot size scales with the bubble score — bigger dots = higher bubble risk.

**Ideal zone:** top-left (high return, low volatility, small dot)  
**Danger zone:** large dots anywhere, especially bottom-right (risk without reward)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
colors_scatter = plt.rcParams['axes.prop_cycle'].by_key()['color']

for i, r in enumerate(watchlist_results):
    vol   = r['volatility']
    ret   = r['total_return']
    score = r['bubble_score']
    c     = colors_scatter[i % len(colors_scatter)]
    size  = max(60, score * 9)

    ax.scatter(vol, ret, s=size, color=c, alpha=0.78,
               edgecolors='black', linewidth=0.8, zorder=4)
    ax.annotate(r['ticker'], (vol, ret),
                xytext=(7, 4), textcoords='offset points',
                fontsize=11, fontweight='bold', color=c)

ax.axhline(0, color='black', lw=0.7, alpha=0.4)
ax.axvline(30, color='#F57C00', lw=1.2, ls='--', alpha=0.6,
           label='High Volatility (30%)')

ax.text(0.02, 0.97, 'Low Risk / High Return\n★ Best Zone',
        transform=ax.transAxes, fontsize=9, color='green', va='top', alpha=0.75)
ax.text(0.68, 0.04, 'High Risk / Low Return\n⚠ Worst Zone',
        transform=ax.transAxes, fontsize=9, color='red', va='bottom', alpha=0.75)

ax.set_xlabel('Annualized Volatility (%)', fontsize=12)
ax.set_ylabel('Total Return (%)', fontsize=12)
ax.set_title('Risk vs Return  (dot size = Bubble Score)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('risk_return_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Interactive CLI

The `main()` function in `stock_bubble_detector.py` runs a full interactive session
where you type a ticker symbol and the program walks through every step automatically.

**To run from the terminal:**
```bash
python stock_bubble_detector.py
```

You will be prompted for a ticker and number of years, then the program:
1. Downloads the data
2. Prints the metric summary
3. Runs bubble analysis and prints the report
4. Prints avoidance strategies
5. Optionally shows the chart
6. Asks if you want to analyze another stock

This cell demonstrates the full pipeline programmatically without user input prompts:

In [ ]:
# Full automated pipeline — same as what main() does interactively
DEMO_TICKER = TICKER
demo_data   = load_stock_data(DEMO_TICKER, START_DATE, END_DATE)

if demo_data is not None:
    # Step 1: Metric summary
    print_stock_summary(DEMO_TICKER, demo_data)

    # Step 2: Bubble analysis
    demo_result = run_bubble_analysis(DEMO_TICKER, demo_data)
    print_bubble_report(demo_result)

    # Step 3: Strategies
    demo_price = get_current_price(demo_data)
    demo_strats = generate_strategies(
        DEMO_TICKER, demo_price, demo_result['bubble_score']
    )
    print_strategies(DEMO_TICKER, demo_result['bubble_score'], demo_strats)

    # Step 4: Chart
    plot_stock_detail(DEMO_TICKER, demo_data)

---
## Conclusion

### Summary of Findings

This notebook demonstrated the complete stock bubble detection pipeline:

| Step | Function | What It Does |
|------|----------|-------------|
| 1 | `load_stock_data()` | Downloads real OHLCV data via Yahoo Finance |
| 2 | `get_volatility()`, `get_rsi()`, etc. | Computes 6 risk metrics |
| 3 | `run_bubble_analysis()` | Scores the stock 0–100 across 4 checks |
| 4 | `generate_strategies()` | Creates risk-scaled action recommendations |
| 5 | `print_bubble_report()` | Formats results for easy reading |
| 6 | `plot_stock_detail()` | Two-panel Price + RSI chart |

### Python Concepts Used

-  Variables, data types, f-strings
-  Lists and dictionaries
-  For loops and conditional logic (`if / elif / else`)
-  Functions with parameters and return values
-  `try / except` error handling
-  pandas: `pct_change()`, `rolling()`, `cummax()`, `replace()`
-  NumPy: `std()`, `sqrt()`
-  Matplotlib: multi-panel figures, `fill_between()`, `axhline()`
-  yfinance: `Ticker()`, `.history()`

### Real-World Relevance

These four bubble indicators are used by:
- **Quantitative hedge funds** — RSI and momentum scoring in algo strategies
- **Risk management desks** at Goldman Sachs, JPMorgan — drawdown and volatility limits
- **Retail platforms** like Robinhood and Fidelity — technical indicator overlays

### Potential Next Steps
- Add **email alerts** when a score crosses 70
- Connect to a **live trading API** (Alpaca, Interactive Brokers)
- Build a **Streamlit web dashboard** with a dropdown for tickers
- **Backtest**: how profitable would selling at score > 70 have been?

---
*Built with Python 3 · yfinance · pandas · numpy · matplotlib*  